# Fusion photo-z — feature set2 (Cathy DCMDN image branch) — GPU-only

Tabular bases are **pre-trained on SciServer** (`SciServerComputeJob/tabular_baseline_job.py`, smart
per-base full-vs-removehard decision) and pulled from GCS — this notebook runs only the GPU work and
logs to MLflow **`fusion-cathy`**:
1. **Load** pre-trained `base_{KEY}_*` + `meta_{KEY}` from gs -> baseline val50k.
2. **Cathy embedding** — DCMDN `fc2_tanh` (100-d), **RAW cutout input** (her training applied no image transform; BN handles it).
3. **Fusion** — MLP+MDN over `[3 base preds | 100 emb]`, early-stop on val50k_sigma_MAD.

Runs: `baseline-set2`, `fusion-set2`. Comparable to Cathy DCMDN alone (val50k 0.01242) on the same 45k val.

In [ ]:
!pip -q install mlflow

In [ ]:
# --- MLflow connect (paste token at the prompt; NOT stored in the notebook) ---
import os, getpass, mlflow, mlflow.tensorflow
os.environ["MLFLOW_TRACKING_URI"] = "https://146-148-10-86.sslip.io"
os.environ["MLFLOW_TRACKING_TOKEN"] = getpass.getpass("MLflow token: ")
EXPERIMENT = "fusion-cathy"
mlflow.set_experiment(EXPERIMENT)
mlflow.tensorflow.autolog(log_models=False, log_datasets=False, checkpoint=False)
print("MLflow ->", os.environ["MLFLOW_TRACKING_URI"], "| experiment:", EXPERIMENT)

In [ ]:
# ====== FEATURE SET + which pre-trained tabular bases to pull (only cell that differs) ======
import numpy as np
FEATURE_SET = "set2-zmag-err-4col-sgsep-conc"; SET_KEY = "set2"     # gs pkls: base_set2_{RF,HGB,MLP}.pkl, meta_set2.pkl
TAB = ["dered_z","modelMagErr_z","u-g","g-r","r-i","i-z","sg_sep","conc_r"]
def build_features(cat):
    col = lambda n: cat[n].mask(cat[n] <= -100).to_numpy("float32")
    du, dg, dr, di, dz = (col(f"dered_{b}") for b in "ugriz")
    R50, R90 = col("petroR50_r"), col("petroR90_r")
    f = {"dered_z":dz,"modelMagErr_z":col("modelMagErr_z"),
         "u-g":np.clip(du-dg,-1,4),"g-r":np.clip(dg-dr,-1,4),"r-i":np.clip(dr-di,-1,4),"i-z":np.clip(di-dz,-1,4),
         "sg_sep":col("psfMag_r")-col("modelMag_r"),"conc_r":R90/np.where(R50==0,np.nan,R50)}
    X = np.stack([f[c] for c in TAB], 1).astype("float32"); X[~np.isfinite(X)] = np.nan
    return X

In [ ]:
# --- pull data + Cathy model + the PRE-TRAINED tabular bases (from SciServer) from GCS ---
from google.colab import auth; auth.authenticate_user()
import os; os.makedirs("/content/data", exist_ok=True)
!gsutil -q cp gs://macrocosm-lewagon/data/sample_v4.5/catalog_v4.parquet /content/data/
!gsutil -q -m cp -n gs://macrocosm-lewagon/data/sample_v4.5/images_*.npy /content/data/
!gsutil -q cp gs://macrocosm-lewagon/data/splits/train_objids.csv gs://macrocosm-lewagon/data/splits/val_objids.csv /content/
!gsutil -q cp gs://macrocosm-lewagon/models/cathy-dcmdn-24-paper-halffilter-run.keras /content/cathy.keras
!gsutil -q cp gs://macrocosm-lewagon/models/base_{SET_KEY}_RF.pkl gs://macrocosm-lewagon/models/base_{SET_KEY}_HGB.pkl gs://macrocosm-lewagon/models/base_{SET_KEY}_MLP.pkl gs://macrocosm-lewagon/models/meta_{SET_KEY}.pkl /content/

import numpy as np, pandas as pd, glob, re, joblib
SHARD = 6000
cat = pd.read_parquet("/content/data/catalog_v4.parquet")
objid = cat["objid"].to_numpy("int64"); z = cat["redshift"].to_numpy("float64"); N = len(cat)
train_ids = set(pd.read_csv("/content/train_objids.csv").objid.astype("int64"))
val_ids   = set(pd.read_csv("/content/val_objids.csv").objid.astype("int64"))
Xall = build_features(cat); keep = ~np.isnan(Xall).any(1)        # complete-feature rows (no mask)
print(f"rows {N:,} | complete-feature {int(keep.sum()):,} | feats {Xall.shape[1]} ({FEATURE_SET}, bases={SET_KEY})")

In [ ]:
# --- metric helpers (no sklearn training here — bases are pre-trained) ---
def smad(zt, zp):
    zt, zp = np.asarray(zt,float), np.asarray(zp,float); d = (zp-zt)/(1+zt)
    return float(1.4826*np.median(np.abs(d-np.median(d))))
def outl(zt, zp, thr=0.05):
    zt, zp = np.asarray(zt,float), np.asarray(zp,float)
    return float(np.mean(np.abs((zp-zt)/(1+zt)) > thr))
ORDER = ["RF","HGB","MLP"]

In [ ]:
# --- load pre-trained tabular bases + LR meta (from SciServer), eval val50k -> MLflow ---
arts  = {m: joblib.load(f"/content/base_{SET_KEY}_{m}.pkl") for m in ORDER}   # {model, remove_hard, features, ...}
bases = {m: arts[m]["model"] for m in ORDER}
flags = {m: arts[m]["remove_hard"] for m in ORDER}
lr    = joblib.load(f"/content/meta_{SET_KEY}.pkl")["model"]

vidx = np.where(keep & np.isin(objid, list(val_ids)))[0]                       # complete-feature val == val50k
Xv, zv = Xall[vidx], z[vidx]
zb = lr.predict(np.column_stack([bases[m].predict(Xv) for m in ORDER]))
base_smad, base_out = smad(zv, zb), outl(zv, zb)*100
print(f"BASELINE val50k: sigma_MAD={base_smad:.5f}  outlier={base_out:.2f}%  (n={len(zv)}) | remove_hard={flags}")
with mlflow.start_run(run_name=f"baseline-{FEATURE_SET}"):
    mlflow.log_params(dict(stage="baseline", feature_set=FEATURE_SET, n_feats=len(TAB),
                           remove_hard=str(flags), source="SciServer tabular_baseline_job"))
    mlflow.log_metric("val50k_sigma_MAD", base_smad); mlflow.log_metric("val50k_outlier", base_out/100)

In [ ]:
# --- Cathy DCMDN embedding: fc2_tanh (100-d), RAW images (no transform; the model's BN handles it) ---
import tensorflow as tf
cnn = tf.keras.models.load_model("/content/cathy.keras", compile=False)        # MDN loss can't deserialize
emb_model = tf.keras.Model(cnn.input, cnn.get_layer("fc2_tanh").output)
D = emb_model.output_shape[-1]; print("Cathy embedder tap fc2_tanh -> dim", D)
paths = {int(re.findall(r"images_(\d+)_", p)[0])//SHARD: p for p in glob.glob("/content/data/images_*.npy")}
emb = np.empty((N, D), "float32")
for s in range(max(paths)+1):
    arr = np.asarray(np.load(paths[s], mmap_mode="r"), "float32")               # (6000,24,24,5) RAW nanomaggies
    emb[s*SHARD : s*SHARD+len(arr)] = emb_model.predict(arr, verbose=0, batch_size=512)
    if s % 25 == 0: print(f"  emb shard {s}")
print("embeddings", emb.shape)

In [ ]:
# --- FUSION: MLP + MDN over [3 base preds | 100 emb] (complete-feature, no mask) ---
from tensorflow.keras import layers as L, Model, Input, regularizers
def mdn_nll(K):
    def loss(y_true, y_pred):
        pi, mu, sig = y_pred[:, :K], y_pred[:, K:2*K], y_pred[:, 2*K:]
        y = tf.expand_dims(y_true, 1)
        lc = (tf.math.log(pi+1e-8) - 0.5*tf.math.log(2*np.pi*sig**2+1e-8) - 0.5*((y-mu)/(sig+1e-8))**2)
        return tf.reduce_mean(-tf.reduce_logsumexp(lc, axis=1))
    return loss
def mdn_point(raw):
    raw = np.asarray(raw); K = raw.shape[1]//3; pi, mu = raw[:, :K], raw[:, K:2*K]
    return mu[np.arange(len(mu)), pi.argmax(1)]
def build_fusion(n_in, mdn=5):
    reg = regularizers.l2(1e-4); inp = Input((n_in,)); x = L.BatchNormalization()(inp)
    for h in (128, 128, 64):
        x = L.Dense(h, activation="relu", kernel_regularizer=reg)(x); x = L.BatchNormalization()(x); x = L.Dropout(0.3)(x)
    pi = L.Dense(mdn, activation="softmax")(x); mu = L.Dense(mdn)(x); sig = L.Dense(mdn, activation="softplus")(x)
    return Model(inp, L.Concatenate(name="z")([pi, mu, sig]))

def fusion_X(idx):
    return np.column_stack([bases[m].predict(Xall[idx]) for m in ORDER] + [emb[idx]]).astype("float32")
tr_idx = np.where(keep & np.isin(objid, list(train_ids)))[0]
Xtr, ytr = fusion_X(tr_idx), np.log1p(z[tr_idx]).astype("float32")
Xva, yva, zva = fusion_X(vidx), np.log1p(z[vidx]).astype("float32"), z[vidx].astype("float64")
print("fusion in-dim", Xtr.shape[1], "| train", Xtr.shape, "| val50k", Xva.shape)

class CB(tf.keras.callbacks.Callback):
    def __init__(self, X, zt): self.X, self.zt = X, zt
    def on_epoch_end(self, e, logs=None):
        sm = smad(self.zt, np.expm1(mdn_point(self.model.predict(self.X, verbose=0))))
        logs = logs or {}; logs["val50k_sigma_MAD"] = sm
        try:
            if mlflow.active_run(): mlflow.log_metric("val50k_sigma_MAD", float(sm), step=e)
        except Exception: pass
        print(f"  epoch {e}: val50k sigma_MAD={sm:.4f}")

K = 5
model = build_fusion(Xtr.shape[1], K)
model.compile(tf.keras.optimizers.Adam(3e-4, clipnorm=1.0), loss=mdn_nll(K))
cbs = [CB(Xva, zva),
       tf.keras.callbacks.EarlyStopping("val50k_sigma_MAD", mode="min", patience=10, restore_best_weights=True),
       tf.keras.callbacks.ReduceLROnPlateau("val50k_sigma_MAD", mode="min", factor=0.5, patience=3, min_lr=1e-5)]
with mlflow.start_run(run_name=f"fusion-{FEATURE_SET}"):
    mlflow.log_params(dict(stage="fusion", feature_set=FEATURE_SET, mdn=K, hidden="128-128-64",
                           inputs=f"{len(ORDER)}base+{emb.shape[1]}emb(cathy fc2_tanh)",
                           baseline_val50k=round(base_smad,5), n_train=len(Xtr)))
    model.fit(Xtr, ytr, validation_data=(Xva, yva), epochs=80, batch_size=1024, callbacks=cbs)
    zp = np.expm1(mdn_point(model.predict(Xva, verbose=0)))
    fus_smad, fus_out = smad(zva, zp), outl(zva, zp)*100
    print(f"\nFUSION val50k: sigma_MAD={fus_smad:.5f}  outlier={fus_out:.2f}%")
    mlflow.log_metric("val50k_sigma_MAD", fus_smad); mlflow.log_metric("val50k_outlier", fus_out/100)
    model.save("/content/fusion.keras"); mlflow.log_artifact("/content/fusion.keras")
print(f"\n=== {FEATURE_SET} ===  baseline {base_smad:.5f}  ->  fusion {fus_smad:.5f}   (Cathy DCMDN alone 0.01242)")